# Python Objects

> 📘 **Python Mastery** · Module 08 — Object-Oriented Programming (OOP) · Lesson 2/5

A class is the blueprint; an **object** (also called an *instance*) is each concrete thing stamped out from it. This lesson is about the object half of object-oriented programming: creating objects, giving each its own state, testing identity, and reaching into attributes dynamically.

## 🎯 Learning Objectives

- Create objects by calling a class like a function
- Demonstrate that every object keeps independent, isolated state
- Read, write, and delete attributes with dot notation and the `hasattr` / `getattr` / `setattr` trio
- Distinguish object **identity** (`is`, `id()`) from **equality** (`==`)
- Explain why "everything in Python is an object" — including classes themselves
- Store and retrieve objects inside collections such as dictionaries

## 1. Instantiating: Calling the Class

You create an object by **calling** the class with parentheses, exactly like a function call. Under the surface two specials cooperate: `__new__` builds the empty object, then `__init__` (Lesson 1) fills in its starting state. Every single call yields a brand-new object.

**Syntax:**

```python
class Wallet:
    def __init__(self, owner):
        self.owner = owner

w = Wallet("Sarah")     # calling Wallet(...) constructs a new object
```

**Example:** three calls, three distinct objects.

In [1]:
class Wallet:
    """A cash wallet with an owner."""

    def __init__(self, owner):
        self.owner = owner
        self.cash = 0

    def __repr__(self):
        return f"Wallet(owner={self.owner!r}, cash={self.cash})"

w1 = Wallet("Sarah")     # call 1 -> object 1
w2 = Wallet("Rafi")      # call 2 -> object 2
w3 = Wallet("Sarah")     # identical DATA, still a third object

print(w1)
print(w2)
print(w3)
print(id(w1) == id(w2), id(w1) == id(w3))    # every object is distinct

Wallet(owner='Sarah', cash=0)
Wallet(owner='Rafi', cash=0)
Wallet(owner='Sarah', cash=0)
False False


## 2. Every Object Owns Its State

Mutating one object never leaks into another: each carries its own copy of the instance attributes. Two bank accounts may share a class — they never share a balance.

**Example:** deposits into one account leave the other untouched.

In [2]:
class BankAccount:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self.balance = balance

    def deposit(self, amount):
        self.balance += amount

    def __str__(self):
        return f"{self.owner} | balance {self.balance}"

sarah_acc = BankAccount("Sarah", 5000)
rafi_acc = BankAccount("Rafi")          # uses the default balance of 0

sarah_acc.deposit(1500)
sarah_acc.deposit(300)                  # only Sarah's object changes

print(sarah_acc)
print(rafi_acc)

Sarah | balance 6800
Rafi | balance 0


## 3. Reading, Writing, and Dot-Chaining

The dot operator reaches inside an object: `obj.attribute` reads, `obj.attribute = value` writes. Because attributes hold real Python values, whatever comes back can immediately be used for more operations — that is **dot-chaining**.

**Example:**

In [3]:
print(sarah_acc.owner)                    # read an attribute
print(sarah_acc.owner.upper())            # it's a real string: chain a method on it
print("  Noise  ".strip().upper())        # chaining works on ANY returned object
print(str(rafi_acc.balance).zfill(6))     # int -> str -> pad with zeros

sarah_acc.owner = "Sarah Ahmed"           # write: replace the value outright
print(sarah_acc.owner)

Sarah
SARAH
NOISE
000000
Sarah Ahmed


## 4. Object Identity: `id()`, `is`, and `==`

Every object has a unique identity number, `id()` (in CPython, effectively its memory address). The `is` operator asks *"same object?"* while `==` asks *"same value?"* — two very different questions. Assigning an object to a second variable copies the **reference**, not the object.

> 🔍 **Under the Hood:** Variable names are bindings to object references. `b = a` binds the name `b` to the *same* object `a` already points at — nothing is duplicated — which is why mutations made through either name are visible through both. For your own classes `==` falls back to identity until you define `__eq__`, so two data-identical accounts still compare unequal.

**Syntax:**

```python
id(obj)        # identity number
a is b         # True iff a and b reference the SAME object
a == b         # True iff values compare equal (your class defines how)
```

**Example:**

In [4]:
class Tag:
    def __init__(self, label):
        self.label = label

a = Tag("box-7")
b = a                        # SECOND NAME for the SAME object
c = Tag("box-7")             # equal-looking, genuinely different object

print(a is b)                # True  -- literally the same object
print(a is c)                # False -- two objects, one coincidence
print(a == c)                # False too: default == compares identity
print(id(a) == id(b), id(a) == id(c))

b.label = "box-8"            # mutate through b...
print(a.label)               # ...and a sees it: same underlying object

True
False
False
True False
box-8


## 5. Everything Is an Object

Integers, strings, lists, functions, modules, and classes — all of them are objects with a type, an identity, and attributes. Even `type` itself is an object whose type is `type`. This uniformity is why Python feels consistent: the same dot-and-method toolbox works on *everything*.

**Example:** attach an attribute to a function, inspect a class.

In [5]:
print(type(42))
print(type("hello"))
print(type([1, 2]))

def cheer(name):
    return f"Go {name}!"

print(type(cheer))            # functions are objects too
cheer.author = "Prof. Ox"     # ...so we can attach attributes to them!
print(cheer.author)

class Dog:
    pass

print(type(Dog))              # a CLASS is an object of type 'type'
print(type(type))             # ...and type's type is type. Turtles all the way down.

<class 'int'>
<class 'str'>
<class 'list'>
<class 'function'>
Prof. Ox
<class 'type'>
<class 'type'>


## 6. `isinstance()`: Asking "Are You One of These?"

`isinstance(obj, Class)` reports whether an object was built from that class — or from any of its subclasses, which makes it **inheritance-aware** (Lesson 3). It accepts a tuple of candidates and is preferred over comparing `type()` results precisely because it respects inheritance.

**Syntax:**

```python
isinstance(obj, SomeClass)
isinstance(obj, (ClassA, ClassB))    # "any of these"
```

**Example:**

In [6]:
class Animal:
    pass

class Dog(Animal):               # a tiny preview of inheritance
    pass

rex = Dog()

print(isinstance(rex, Dog))
print(isinstance(rex, Animal))          # True: inheritance-aware
print(isinstance("hello", str))
print(isinstance(3.14, (int, float)))   # tuple = "any of these"
print(isinstance(rex, (int, str)))
print(type(rex) == Dog)                 # works, but ignores subclasses: prefer isinstance

True
True
True
True
False
True


## 7. Dynamic Attributes: `hasattr`, `getattr`, `setattr`

An attribute name is just a string underneath, so Python lets you probe, read, and create attributes **by string at runtime**. This trio is the engine behind plugin systems, ORMs, and config-driven scripts.

**Syntax:**

```python
hasattr(obj, "name")                 # does the attribute exist?
getattr(obj, "name", default)        # read, with a safe fallback
setattr(obj, "name", value)          # create or overwrite
```

**Example:**

In [7]:
class Product:
    def __init__(self, name, price):
        self.name = name
        self.price = price

item = Product("Mechanical Keyboard", 89.99)

print(hasattr(item, "price"), hasattr(item, "discount"))

print(getattr(item, "price"))
print(getattr(item, "discount", 0.0))     # missing -> default instead of a crash

setattr(item, "discount", 0.15)           # creates the attribute on the fly
print(item.discount)

for field in ("name", "price", "discount"):
    print(field, "=", getattr(item, field))

True False
89.99
0.0
0.15
name = Mechanical Keyboard
price = 89.99
discount = 0.15


In [8]:
# Practical pattern: build attributes from a dictionary of settings
settings = {"theme": "dark", "font_size": 14}

class AppSettings:
    pass

app = AppSettings()
for key, value in settings.items():
    setattr(app, key, value)              # attribute names come from data

print(app.theme, app.font_size)
print(vars(app))                          # peek at the object's raw attribute dict

dark 14
{'theme': 'dark', 'font_size': 14}


## 8. Deleting Attributes and Objects

`del obj.attr` removes one attribute from an object; `del obj` removes the *variable name*. If that was the last reference to the object, Python's memory manager reclaims it automatically — you never free memory by hand.

**Syntax:**

```python
del obj.attr     # remove one attribute
del obj          # remove the NAME (binding), not necessarily the object
```

**Example:**

In [9]:
class Product:
    def __init__(self, name, price):
        self.name = name
        self.price = price

item = Product("USB-C Cable", 12.0)
item.temp_note = "clearance"              # add a throwaway attribute

del item.temp_note
print(hasattr(item, "temp_note"))         # False -- gone

del item                                  # delete the NAME, not other names
try:
    print(item)
except NameError as err:
    print("NameError:", err)

False
NameError: name 'item' is not defined


## 9. Attribute Shadowing

Reading `obj.attr` searches the **instance first**, then the class. So assigning through an instance creates an instance attribute that temporarily *shadows* the class attribute — and deleting it lets the class value shine through again.

**Syntax:**

```python
obj.attr = value    # creates/replaces the INSTANCE attribute (shadows the class one)
del obj.attr        # drop the shadow; the class attribute is visible again
```

**Example:**

In [10]:
class Bird:
    can_fly = True                 # class-level default for all birds

    def __init__(self, name):
        self.name = name

tweety = Bird("Tweety")
pingu = Bird("Pingu")

pingu.can_fly = False              # instance attribute SHADOWS the class one
print(tweety.can_fly)              # True  -- untouched class default
print(pingu.can_fly)               # False -- the instance copy wins

del pingu.can_fly                  # remove the shadow...
print(pingu.can_fly)               # ...and the class value is visible again

True
False
True


## 10. Objects as Dictionary Values: a Mini Contact Book

Collections happily store objects — this is how real applications model records, rows, and entities. Here a dictionary maps usernames to `Contact` objects, combining Lesson 1's blueprint with this lesson's storage.

**Example:**

In [11]:
class Contact:
    """One address-book entry."""

    def __init__(self, name, phone, city):
        self.name = name
        self.phone = phone
        self.city = city

    def __str__(self):
        return f"{self.name} ({self.phone}) - {self.city}"

contacts = {
    "sarah": Contact("Sarah Ahmed", "01711-223344", "Dhaka"),
    "rafi":  Contact("Rafi Chowdhury", "01822-556677", "Chattogram"),
    "amina": Contact("Amina Karim", "01933-889900", "Sylhet"),
}

print(contacts["sarah"])                       # fetch the object, then use it
print(contacts["rafi"].phone)                  # chain into its attributes
print(contacts["amina"].city.upper())

for handle, entry in contacts.items():
    print(f"@{handle}: {entry}")

Sarah Ahmed (01711-223344) - Dhaka
01822-556677
SYLHET
@sarah: Sarah Ahmed (01711-223344) - Dhaka
@rafi: Rafi Chowdhury (01822-556677) - Chattogram
@amina: Amina Karim (01933-889900) - Sylhet


## ⚠️ Common Mistakes & Gotchas

| Mistake | Problem | Fix |
|---|---|---|
| Comparing values with `is` (as in `total is 1000`) | `is` checks identity, not equality — results surprise you | Use `==` for values; reserve `is` for `None` and singletons |
| Thinking `b = a` copies the object | It copies the reference; mutations leak through both names | Need an independent snapshot? `copy.deepcopy(a)` |
| Reading a possibly-missing attribute directly | `AttributeError` crashes the flow | Guard with `hasattr`, or `getattr(obj, name, default)` |
| "Fixing" a shared class constant via `obj.CONSTANT = ...` | Only that one instance diverges — a shadowing trap | Update class attributes through the class name |
| Expecting `del obj` to vaporize the object instantly | Other names may still reference it; cleanup waits for garbage collection | Understand references; let Python manage memory |

## 💡 Best Practices & Pro Tips

- Build **fully formed** objects: pass all starting state through `__init__` instead of patching attributes afterward.
- Write `if x is None:` — the idiomatic (and correct) use of `is`.
- Prefer `isinstance()` over `type(x) == Y`: it survives refactoring and honors subclasses.
- Objects in lists/dicts beat parallel lists for anything with more than one field — one `Contact` instead of three synchronized lists.
- 🤖 **AI-engineering relevance:** data science is attribute access all the way down — `df.shape`, `df.columns`, `model.coef_`. And the config-driven pattern `getattr(args, "learning_rate", 3e-4)` appears in almost every ML research repo you will ever read.

## 📌 Summary

| Expression | What it does | Example |
|---|---|---|
| `ClassName(args)` | Constructs a new object | `Wallet("Sarah")` |
| `obj.attr` / `obj.attr = v` | Read / write an attribute | `acc.balance` |
| `id(obj)` | Identity number (CPython: memory address) | `id(acc)` |
| `a is b` | Same object? | `a is b` |
| `a == b` | Same value? | `a == c` |
| `isinstance(o, C)` | Built from C or a subclass? | `isinstance(rex, Dog)` |
| `hasattr` / `getattr` / `setattr` | String-based attribute toolkit | `getattr(cfg, "lr", 3e-4)` |
| `del obj.attr` / `del obj` | Remove an attribute / remove a name | `del item.temp_note` |
| `vars(obj)` | Instance attributes as a dict | `vars(app)` |

**Key takeaways**
- Calling a class builds a fresh object; no two constructions ever share state.
- `is` = same object, `==` = same value — conflating them causes subtle bugs.
- Names bind to objects; assignment never copies.
- Attribute shadowing means instance assignments hide class defaults until deleted.

> 🔗 **Next Lesson:** [03 · Python Inheritance](../03_Inheritance/) — let classes borrow, reuse, and specialize each other without copying a single line.